# Lab 05: How to rebuild a RAG pipeline with LangChain

**CPU | API key: `OPENAI_API_KEY` (chat and embedding calls) | Instructor demo**

## Goal

Rebuild the pipeline of Labs 01 and 02 with LangChain, and check question by question that it gives the same results
as your own code. The moment that teaches it: both versions find the right paragraph for every question, yet they
hand the answer step different chunks for every question. A LangChain default returns one chunk more than you asked
for in Lab 01, and the overall score could not show it.

## What you will learn

- How to turn chunk records into LangChain `Document`s that keep their IDs and fields.
- How LangChain's embeddings and chat model are built from `models.env` (`lc_embeddings()`, `lc_chat_model()`), and
  what `OpenAIEmbeddings` sends to the API.
- How to index documents in `InMemoryVectorStore` and search them with scores.
- How to check a framework's results against your own, question by question, and find the default that made them
  differ.
- How to get a cited answer with `with_structured_output` and check it with Lab 02's quote check.
- How to leave documents out of a search with a metadata filter.

**In class:** the instructor shows Parts A, C and D before Lab 06. Parts B and E to H are self-study. To follow along,
run Part B's cells too, because Part C uses the embeddings Part B builds.

## Set up the notebook

Run these two cells first, every time; Lab 00 explains each line. **Setup** finds the repository folder (on Colab it
copies the repository and installs the packages first). **Config** reads the settings and finds your key.

In [1]:
# Setup: on Colab, copy the repository and install the packages. Everywhere else they are installed already.
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    if not Path("/content/rag_for_nlp").exists():
        !git clone -q --depth 1 https://github.com/tatwan/rag_for_nlp.git /content/rag_for_nlp
    %cd /content/rag_for_nlp
    %pip install -q -r requirements.txt

ROOT = next(folder for folder in [Path.cwd(), *Path.cwd().parents] if (folder / "models.env").exists())
sys.path.insert(0, str(ROOT))
print("Repository folder:", ROOT.name)

Repository folder: rag_for_nlp


In [2]:
# Config: settings from models.env (your .env and environment variables win); the key from the environment, .env or Colab Secrets.
from shared.settings import load_settings, load_key, stamp

settings = load_settings()
key_found_in = load_key("OPENAI_API_KEY")

for name, value in stamp(settings).items():
    print(f"{name:18}{value}")
print(f"{'OPENAI_API_KEY':18}{'found in ' + key_found_in if key_found_in else 'not found'}")

chat_model        gpt-4o-mini
reasoning_effort  None
embed_model       text-embedding-3-small
judge_model       gpt-4o-mini
rerank_model      gpt-4o-mini
chat_endpoint     https://api.openai.com/v1
embed_endpoint    https://api.openai.com/v1
run_date          2026-10-24
OPENAI_API_KEY    found in .env


This lab uses `chat_model` and `embed_model` from the lines above, and `JSON_MODE` and `TOKEN_ENCODING` from
`models.env`. It makes about 110 short chat calls and embeds the corpus two or three times (about 21,000 tokens each),
well under one US cent on `gpt-4o-mini`. The saved outputs and every count in the text come from `gpt-4o-mini`; the
README says what changed with `gpt-6-luna`. Your own run will differ a little: model replies vary from run to run and similarity scores can move in the third decimal, so compare the order of results and the kind of reply with the text, not every number.

The labs before this one built every step in plain Python. From here on the labs use
[LangChain](https://docs.langchain.com/oss/python/langchain/overview) for the models, the documents, the vector store
and structured answers. Before you build on it, you check it against your own pipeline on the same questions. In the
printed output, *plain* is your pipeline from Labs 01 and 02 and *LangChain* the new one.

---

## Part A: Turn the chunks into documents

Part A gives LangChain the corpus in the shape its components take.

Load the same corpus as Lab 02: the 272 chunks and the five product policies, each policy as one chunk whose ID is its
source ID.

In [3]:
import json

DATA = ROOT / "data"
chunks = json.loads((DATA / "squad-chunks.json").read_text(encoding="utf-8"))
policies = json.loads((DATA / "support-case.json").read_text(encoding="utf-8"))
corpus = chunks + [{**policy, "chunk_id": policy["source_id"]} for policy in policies]
print(len(corpus), "records")
print("a chunk: ", list(corpus[0]))
print("a policy:", list(corpus[-1]))

277 records
a chunk:  ['chunk_id', 'source_id', 'title', 'start', 'end', 'text', 'tokens']
a policy: ['source_id', 'title', 'text', 'version', 'access', 'chunk_id']


Every LangChain component that reads text takes a **`Document`**
([reference](https://reference.langchain.com/python/langchain-core/documents/base/Document)): `page_content`, the text a
store embeds and a model reads; `metadata`, a dictionary for everything else; and an `id`. Build one by hand from the
first record:

In [4]:
from langchain_core.documents import Document

record = corpus[0]
document = Document(id=record["chunk_id"], page_content=record["text"],
                    metadata={name: value for name, value in record.items() if name != "text"})
print("id:      ", document.id)
print("metadata:", document.metadata)
print("text:    ", document.page_content[:80], "...")

id:       S00-P00-C00
metadata: {'chunk_id': 'S00-P00-C00', 'source_id': 'S00-P00', 'title': 'Super_Bowl_50', 'start': 0, 'end': 486, 'tokens': 98}
text:     Super Bowl 50 was an American football game to determine the champion of the Nat ...


The text moved to `page_content` and every other field to `metadata`. The `id` matters most: a store keys each
document by it, and a citation names it. A document without one gets a random ID from the store. The metadata keeps
what later steps read: `source_id` for hit@3, and on the policies `version` and `access`. Now all 277:

In [5]:
documents = [Document(id=record["chunk_id"], page_content=record["text"],
                      metadata={name: value for name, value in record.items() if name != "text"})
             for record in corpus]
print(len(documents), "documents")
print(documents[-1])

277 documents
page_content='Atlas invoices are sent monthly. Workspace owners can download invoices from Billing.' metadata={'source_id': 'H05', 'title': 'Atlas billing', 'version': '2026-10-01', 'access': 'public', 'chunk_id': 'H05'}


### Checkpoint A

| You should see | Our run |
|---|---|
| Corpus | 277 records; chunks with `start`, `end` and `tokens`, policies with `version` and `access` |
| Documents | 277; the last is H05, with its title, `version`, `access: 'public'` and `chunk_id` in the metadata |

---

## Part B: Build the embeddings from the settings (self-study)

Part B gives LangChain the embedding model Lab 01 used, and checks that it returns the same vector.

`OpenAIEmbeddings`, from the `langchain-openai` package, calls the same Embeddings API as Lab 01's client. Build it by
hand from the settings, embed the Tesla question, and compare the vector with the one Lab 01's `embed()` returns:

In [6]:
import os
import numpy as np
from langchain_openai import OpenAIEmbeddings
from shared.settings import embed_url
from shared.search import embed

embeddings = OpenAIEmbeddings(model=settings["EMBED_MODEL"], base_url=embed_url(settings),
                              api_key=os.environ["OPENAI_API_KEY"])
question = "In what year was Nikola Tesla born?"
lc_vector = np.array(embeddings.embed_query(question))
plain_vector = embed([question], settings)[0]
print("numbers:", len(lc_vector), "| largest difference from embed():", f"{np.abs(lc_vector - plain_vector).max():.6f}")

numbers: 1536 | largest difference from embed(): 0.000000


The same model gives the same vector: 1,536 numbers, and the largest difference is 0.000000. `embed_query()` embeds one
question, `embed_documents()` a list.

A framework can still send something other than what you wrote. Wrap the SDK method LangChain calls,
`embeddings.client.create`, so that it prints its input before sending it:

In [7]:
send = embeddings.client.create

def show_input(**request):
    print("input sent:", request["input"])
    return send(**request)

embeddings.client.create = show_input
embeddings.embed_query(question)
embeddings.client.create = send          # put the original back

input sent: [[644, 1148, 1060, 574, 97415, 28298, 9405, 30]]


The input is a list of numbers, not your text: **token IDs**. LangChain first cuts every text into tokens with
`tiktoken`, so that it can split a text longer than the model's limit into parts and average their vectors. OpenAI
accepts token IDs; many other OpenAI-compatible servers accept only text and reject the request. The switch is
`check_embedding_ctx_length`, `True` by default
([OpenAIEmbeddings reference](https://reference.langchain.com/python/langchain-openai/embeddings/base/OpenAIEmbeddings)).

`lc_embeddings()` in `shared/settings.py` builds this object from the settings, with `EMBED_API_KEY` when
`EMBED_BASE_URL` points to another provider, and turns the switch off for any endpoint that is not OpenAI's. Read it,
then use it:

In [8]:
import inspect
from shared.settings import lc_embeddings

print(inspect.getsource(lc_embeddings))
embeddings = lc_embeddings(settings)
print("send token IDs:", embeddings.check_embedding_ctx_length)

def lc_embeddings(settings: dict | None = None):
    """The same embedding model as embed_client(), as LangChain OpenAIEmbeddings (Lab 05 on).

    LangChain splits long texts with tiktoken and sends token IDs by default; other OpenAI-compatible servers
    may reject token IDs, so that is switched off whenever the endpoint is not OpenAI's.
    """
    from langchain_openai import OpenAIEmbeddings
    settings = settings or load_settings()
    key = os.environ.get("OPENAI_API_KEY")
    if settings.get("EMBED_BASE_URL") and load_key("EMBED_API_KEY"):
        key = os.environ["EMBED_API_KEY"]
    return OpenAIEmbeddings(model=settings["EMBED_MODEL"], base_url=embed_url(settings), api_key=key,
                            check_embedding_ctx_length=embed_url(settings) == OPENAI_DEFAULT_URL)

send token IDs: True


On OpenAI's endpoint it keeps the token IDs, which give the same vectors as text. The labs use it from here on.

### Checkpoint B

| You should see | Our run |
|---|---|
| The vector | 1,536 numbers, largest difference from `embed()` 0.000000 |
| What is sent | `input sent: [[644, 1148, 1060, ...]]`, the question as token IDs |
| `lc_embeddings()` | `send token IDs: True` on `https://api.openai.com/v1` |

---

## Part C: Index the documents and search with scores

Part C puts the 277 documents in a vector store and searches one question with both pipelines.

A **vector store** keeps each document with its vector and ranks the documents for a query.
[`InMemoryVectorStore`](https://reference.langchain.com/python/langchain-core/vectorstores/in_memory/InMemoryVectorStore),
from `langchain-core`, keeps them in a Python dictionary and ranks by cosine similarity: Lab 01's search behind
LangChain's interface. Stores backed by a database (pgvector, Qdrant and others) have the same methods.
`add_documents()` embeds every document and stores it under its ID:

In [9]:
from langchain_core.vectorstores import InMemoryVectorStore

store = InMemoryVectorStore(embeddings)
ids = store.add_documents(documents)
print(len(ids), "documents stored | first IDs:", ids[:2], "| last:", ids[-2:])

277 documents stored | first IDs: ['S00-P00-C00', 'S00-P00-C01'] | last: ['H04', 'H05']


The store kept the documents' own IDs. Now search the Tesla question both ways: your `search()` over Lab 02's cached
vectors (embedded again if your `.cache/` folder is empty), and the store's `similarity_search_with_score()`, called the
way LangChain's [semantic search tutorial](https://docs.langchain.com/oss/python/langchain/knowledge-base) calls it.
Predict first: the same chunks?

In [10]:
from shared.search import chunk_vectors, search

vectors = chunk_vectors(corpus, settings)
print("plain: search()")
for score, chunk in search(question, corpus, vectors, settings):
    print(f"  {score:.4f}  {chunk['chunk_id']}")
print("LangChain: store.similarity_search_with_score()")
for document, score in store.similarity_search_with_score(question):
    print(f"  {score:.4f}  {document.id}")

plain: search()


  0.7216  S03-P00-C00
  0.5404  S03-P01-C00
  0.4655  S03-P02-C00
LangChain: store.similarity_search_with_score()


  0.7221  S03-P00-C00
  0.5405  S03-P01-C00
  0.4656  S03-P02-C00
  0.2592  S03-P01-C01


The same three chunks come first, in the same order, with scores less than a thousandth apart in our run. The store embedded the chunks again, and Lab 01 measured
differences of a few thousandths between two embeddings. This store's score is the cosine similarity, so higher is closer; other stores return a distance,
where lower is closer, so check what a store's score means before you compare scores. And the store printed four rows
where `search()` printed three. Part D asks whether that changes the measured result.

### Checkpoint C

| You should see | Our run |
|---|---|
| Stored | 277 documents, IDs from `S00-P00-C00` to `H05` |
| `search()` | `S03-P00-C00` about 0.72, `S03-P01-C00` 0.54, `S03-P02-C00` 0.47 |
| The store | the same three chunks with the same scores to two decimals, then a fourth row, `S03-P01-C01` about 0.26 |

---

## Part D: Check the two searches on 30 questions

Part D is the parity check for the search: the same 30 questions and the same measure, in both pipelines.

Start with Lab 01's measure, known-source hit@3. Your pipeline has `hit_at_k()`. For the store, a question is a hit
when any document it returns comes from a paragraph in the question's `known_support`, read from the metadata:

In [11]:
from shared.search import hit_at_k

questions = json.loads((DATA / "development-questions.json").read_text(encoding="utf-8"))
plain_hits = hit_at_k(questions, corpus, vectors, settings)

def is_hit(question, hits):
    return any(doc.metadata["source_id"] in question["known_support"] for doc, _ in hits)

lc_results = [store.similarity_search_with_score(q["question"]) for q in questions]
lc_hits = sum(is_hit(q, hits) for q, hits in zip(questions, lc_results))
print(f"hit@3: plain {sum(plain_hits)} of {len(questions)} | LangChain {lc_hits} of {len(questions)}")

hit@3: plain 30 of 30 | LangChain 30 of 30


30 and 30. That looks like parity, but a total can agree while the two pipelines return different
things. Compare what each returned, question by question: the same chunk IDs in the same order.

In [12]:
plain_results = [search(q["question"], corpus, vectors, settings) for q in questions]
plain_ids = [[chunk["chunk_id"] for _, chunk in hits] for hits in plain_results]
lc_ids = [[doc.id for doc, _ in hits] for hits in lc_results]
print(f"same chunks in the same order: {sum(a == b for a, b in zip(plain_ids, lc_ids))} of {len(questions)}")
print("first question | plain:", plain_ids[0], "| LangChain:", lc_ids[0])

same chunks in the same order: 0 of 30
first question | plain: ['S00-P00-C00', 'S00-P01-C01', 'S00-P02-C00'] | LangChain: ['S00-P00-C00', 'S00-P01-C01', 'S00-P02-C00', 'S00-P01-C00']


0 of 30. The first question shows why: the LangChain list has a fourth chunk. You never asked for four, so it
comes from a **default**, the value a function uses for an argument you do not pass. A function's signature lists its
defaults:

In [13]:
print(inspect.signature(store.similarity_search_with_score))

(query: 'str', k: 'int' = 4, **kwargs: 'Any') -> 'list[tuple[Document, float]]'


`k: int = 4`: the store returns four documents unless you pass `k`; your `search()` returns three. So the LangChain
"hit@3" above was really hit@4, and none of these 30 questions needed the fourth chunk, so the count could not show
the difference. The answers would: the model reads a fourth chunk every time. Count the evidence tokens over the 30
questions, with four chunks and with the first three:

In [14]:
import tiktoken

encoding = tiktoken.get_encoding(settings["TOKEN_ENCODING"])

def evidence_tokens(hits):
    return sum(len(encoding.encode(doc.page_content)) for doc, _ in hits)

four = sum(evidence_tokens(hits) for hits in lc_results)
three = sum(evidence_tokens(hits[:3]) for hits in lc_results)
print(f"evidence tokens over 30 questions: k=4 {four} | k=3 {three} | {four / three - 1:.0%} more")

evidence tokens over 30 questions: k=4 9946 | k=3 7764 | 28% more


9,946 tokens against 7,764: 28% more input to every answer, for the same hit@3 on this set. Set
`k=3` and check again, with the largest score difference between the two pipelines:

In [15]:
lc_results = [store.similarity_search_with_score(q["question"], k=3) for q in questions]
lc_ids = [[doc.id for doc, _ in hits] for hits in lc_results]
lc_hits = sum(is_hit(q, hits) for q, hits in zip(questions, lc_results))
gap = max(abs(plain_score - lc_score)
          for plain, lc in zip(plain_results, lc_results)
          for (plain_score, _), (_, lc_score) in zip(plain, lc))
print(f"hit@3: LangChain {lc_hits} of {len(questions)}"
      f" | same chunks in the same order: {sum(a == b for a, b in zip(plain_ids, lc_ids))} of {len(questions)}")
print(f"largest score difference: {gap:.4f}")
print(f"\n{settings['EMBED_MODEL']} at {embed_url(settings)}")

hit@3: LangChain 30 of 30 | same chunks in the same order: 30 of 30
largest score difference: 0.0005

text-embedding-3-small at https://api.openai.com/v1


Same chunks in the same order for 30 of 30 questions, and no score more than a few thousandths apart. The search matches.

Two habits come out of this part. Compare two pipelines question by question, not by a total: the total agreed before
the lists did. And pass every value your result depends on, even when the default looks harmless: `k` is the number
of chunks every answer reads.

### Checkpoint D

| You should see | Our run |
|---|---|
| hit@3 at the store's default | plain 30 of 30, LangChain 30 of 30 |
| Same lists | 0 of 30; the signature shows `k: int = 4` |
| Evidence tokens | k=4 9,946, k=3 7,764 (28% more) |
| With `k=3` | hit@3 30 of 30, same lists 30 of 30, largest score difference a few thousandths at most |

---

## Part E: Get a cited answer with `with_structured_output` (self-study)

Part E rebuilds Lab 02's answer step: the chat model, the answer's schema, and a reply your code can check.

`ChatOpenAI`, from `langchain-openai`, calls Chat Completions. Build it by hand from the settings and send one message:

In [16]:
from langchain_openai import ChatOpenAI
from shared.settings import chat_url

chat = ChatOpenAI(model=settings["CHAT_MODEL"], base_url=chat_url(settings), api_key=os.environ["OPENAI_API_KEY"],
                  use_responses_api=False)
reply = chat.invoke("In one sentence, what does a vector store do?")
print(type(reply).__name__, "|", reply.content)
print(reply.usage_metadata)

AIMessage | A vector store manages and retrieves high-dimensional vectors, enabling efficient similarity search and retrieval of related items, such as text or images, based on their vector representations.
{'input_tokens': 18, 'output_tokens': 32, 'total_tokens': 50, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}}


`invoke()` takes a string or a list of messages and returns an **`AIMessage`**: the reply in `content`, and the tokens
billed in `usage_metadata`, Lab 02's `usage`. `use_responses_api=False` keeps `ChatOpenAI` on Chat Completions; left
out, it may pick OpenAI's newer Responses API, which other OpenAI-compatible servers usually do not offer.

This call sent no `temperature`, like Lab 02's first call. `lc_chat_model()` adds what `chat_options()` adds in Lab 02:
`temperature`, or `reasoning_effort` for a reasoning model. Read it, then use it:

In [17]:
from shared.settings import lc_chat_model

print(inspect.getsource(lc_chat_model))
chat = lc_chat_model("CHAT_MODEL", settings)
print("model:", chat.model_name, "| temperature:", chat.temperature, "| reasoning_effort:", chat.reasoning_effort)

def lc_chat_model(name: str = "CHAT_MODEL", settings: dict | None = None, temperature: float = 0):
    """The same chat model as chat_client() + chat_options(), as a LangChain ChatOpenAI (Lab 05 on).

    use_responses_api=False keeps it on Chat Completions, which every compatible server implements.
    """
    from langchain_openai import ChatOpenAI
    settings = settings or load_settings()
    options = chat_options(name, settings, temperature)
    return ChatOpenAI(base_url=chat_url(settings), api_key=os.environ.get("OPENAI_API_KEY"),
                      use_responses_api=False, **options)

model: gpt-4o-mini | temperature: 0.0 | reasoning_effort: None


Next, the schema. In Lab 02 you wrote it as a dictionary and checked each reply's shape with `parse()`. LangChain's
[structured output](https://docs.langchain.com/oss/python/langchain/models#structured-output) takes a **Pydantic** class
instead: a Python class whose typed fields are the schema. Pydantic checks a reply when it builds the object and raises
an error when a field is missing or has the wrong type, so the class also does the work of `parse()`. Write the
answer's shape as two classes and compare their fields with Lab 02's `SCHEMA`:

In [18]:
from pydantic import BaseModel
from shared import answer as answering

class Claim(BaseModel):
    claim: str
    source_id: str
    quote: str

class CitedAnswer(BaseModel):
    answer: str
    claims: list[Claim]

schema = CitedAnswer.model_json_schema()
print("Pydantic:", schema["required"], schema["$defs"]["Claim"]["required"])
print("Lab 02:  ", answering.SCHEMA["required"], answering.SCHEMA["properties"]["claims"]["items"]["required"])

Pydantic: ['answer', 'claims'] ['claim', 'source_id', 'quote']
Lab 02:   ['answer', 'claims'] ['claim', 'source_id', 'quote']


The same fields. The model reads evidence lines in Lab 02's format, `[ID] text`, and the quote check reads the same
lines. Write them from the store's hits, with `k=3`:

In [19]:
def evidence(hits):
    return [f"[{doc.id}] {doc.page_content}" for doc, _ in hits]

lines = evidence(store.similarity_search_with_score(question, k=3))
for line in lines:
    print(line[:90], "...")

[S03-P00-C00] Nikola Tesla (Serbian Cyrillic: Никола Тесла; 10 July 1856 – 7 January 1943) ...
[S03-P01-C00] Tesla gained experience in telephony and electrical engineering before emigr ...
[S03-P02-C00] Tesla went on to pursue his ideas of wireless lighting and electricity distr ...


`with_structured_output(CitedAnswer)` returns a new object built on the model; `JSON_MODE=json_object` maps to
`method="json_mode"`, as in Lab 02. Build it and look at what it holds:

In [20]:
method = "json_schema" if settings["JSON_MODE"] == "json_schema" else "json_mode"
structured = chat.with_structured_output(CitedAnswer, method=method)
print(type(structured).__name__, "| step 1 sends response_format =", structured.first.kwargs["response_format"].__name__)

RunnableSequence | step 1 sends response_format = CitedAnswer


A **`RunnableSequence`**, LangChain's name for steps run one after another: first the model, with your class bound to
every call as the `response_format` (OpenAI's Structured Outputs, Lab 02's `json_schema`); then a parser that builds a
`CitedAnswer` from the reply. Ask the Tesla question with Lab 02's rules, `shared.answer.RULES`, abstention line
included:

In [21]:
def messages_for(question, lines):
    return [("system", answering.RULES), ("user", f"Question: {question}\nEvidence:\n" + "\n".join(lines))]

reply = structured.invoke(messages_for(question, lines))
print(type(reply).__name__, "|", reply.answer)
for claim in reply.claims:
    print(f"  quote found: {answering.quote_found(claim.model_dump(), lines)} | {claim.source_id} | {claim.quote}")

CitedAnswer | Nikola Tesla was born in the year 1856.
  quote found: True | S03-P00-C00 | 10 July 1856


The reply is a `CitedAnswer` object, not text: *Nikola Tesla was born in the year 1856.*, with one claim from `S03-P00-C00` (*10 July 1856*) whose quote is found.
`claim.model_dump()` turns a claim into the dictionary Lab 02's `quote_found()` reads, so the check is unchanged.

### Checkpoint E

| You should see | Our run |
|---|---|
| Hand-built model | an `AIMessage`, with `usage_metadata` |
| `lc_chat_model()` | `temperature: 0.0`, `reasoning_effort: None` on `gpt-4o-mini` |
| Schema | the same required fields as Lab 02's `SCHEMA` |
| `structured` | `RunnableSequence`, `response_format = CitedAnswer` |
| Tesla | *Nikola Tesla was born in the year 1856.*, quote found: `True` |

---

## Part F: Check the two answer steps on 30 questions (self-study)

Part F finishes the parity check: the same evidence lines go to Lab 02's `answer()` and to the LangChain model, for
all 30 development questions, one answer each.

In [22]:
INSUFFICIENT = "Insufficient evidence."
all_lines, plain_replies, lc_replies = [], [], []
for q in questions:
    q_lines = evidence(store.similarity_search_with_score(q["question"], k=3))
    all_lines.append(q_lines)
    plain_replies.append(answering.answer(q["question"], q_lines, settings))
    lc_replies.append(structured.invoke(messages_for(q["question"], q_lines)).model_dump())
print(len(plain_replies), "answers from each pipeline")

30 answers from each pipeline


`model_dump()` turned each LangChain answer into the dictionary Lab 02's `answer()` returns, so one function can count
both. It counts Lab 02's measures: refusals (these 30 questions should get none), claims and quotes found.

In [23]:
def summary(replies):
    refused = sum(reply["answer"].strip() == INSUFFICIENT for reply in replies)
    claims = [(q_lines, claim) for q_lines, reply in zip(all_lines, replies) for claim in reply["claims"]]
    found = sum(answering.quote_found(claim, q_lines) for q_lines, claim in claims)
    return f"refused {refused} of {len(replies)} | claims {len(claims)} | quotes found {found}"

def cited(reply):
    return [claim["source_id"] for claim in reply["claims"]]

print("plain:    ", summary(plain_replies))
print("LangChain:", summary(lc_replies))
print("same cited IDs:", sum(cited(p) == cited(l) for p, l in zip(plain_replies, lc_replies)), "of 30",
      "| same answer text:", sum(p["answer"] == l["answer"] for p, l in zip(plain_replies, lc_replies)), "of 30")

plain:     refused 0 of 30 | claims 30 | quotes found 30
LangChain: refused 0 of 30 | claims 31 | quotes found 31
same cited IDs: 29 of 30 | same answer text: 25 of 30


Both pipelines refuse 0 of 30 and every quote is found: 30 claims plain, 31 LangChain. The cited IDs agree for
29 of 30 questions; the answer text agrees for 25 of 30. Print the answers whose text or cited IDs differ, or that
have a quote the check did not find (printed after the answer):

In [24]:
def not_found(reply, q_lines):
    return [claim["quote"] for claim in reply["claims"] if not answering.quote_found(claim, q_lines)]

for plain, lc, q_lines in zip(plain_replies, lc_replies, all_lines):
    if plain["answer"] != lc["answer"] or cited(plain) != cited(lc) or not_found(plain, q_lines) + not_found(lc, q_lines):
        print("plain:    ", cited(plain), plain["answer"], not_found(plain, q_lines) or "")
        print("LangChain:", cited(lc), lc["answer"], not_found(lc, q_lines) or "", "\n")

plain:     ['S00-P00-C00'] The Denver Broncos represented the AFC at Super Bowl 50. 
LangChain: ['S00-P00-C00'] The AFC was represented by the Denver Broncos at Super Bowl 50.  

plain:     ['S00-P01-C00'] Quarterback Cam Newton was named the NFL Most Valuable Player (MVP). 
LangChain: ['S00-P01-C00'] Cam Newton was named the NFL Most Valuable Player.  

plain:     ['S01-P00-C01'] 1745 
LangChain: ['S01-P00-C01'] Casimir Pulaski was born in 1745.  

plain:     ['S02-P00-C00'] Normandy is located in France. 
LangChain: ['S02-P00-C00', 'S02-P01-C01'] Normandy is located in France.  

plain:     ['S02-P02-C00'] The original meaning of the word Norman is "Northman" or "Norseman, Viking." 
LangChain: ['S02-P02-C00'] The original meaning of the word Norman is "Norseman, Viking."  

plain:     ['S04-P02-C00'] The field of computer science that analyzes the resource requirements of a specific algorithm isolated unto itself within a given problem is the analysis of algorithms. 
LangChain: ['S04

Five of the six state the same fact from the same chunk in other words, such as *1745* against *Casimir Pulaski
was born in 1745.*; the largest difference is the Norman answer, where LangChain's leaves out *Northman*. In the
sixth, *Normandy is located in France.*, LangChain also cites `S02-P01-C01`. Are these differences the framework,
or the model? Ask Lab 02's `answer()` the same 30 questions once more and compare it
with its own first run:

In [25]:
again = [answering.answer(q["question"], q_lines, settings) for q, q_lines in zip(questions, all_lines)]
print("plain, second run:", summary(again))
print("same answer text as the first run:", sum(a["answer"] == p["answer"] for a, p in zip(again, plain_replies)), "of 30")

plain, second run: refused 0 of 30 | claims 30 | quotes found 30
same answer text as the first run: 28 of 30


Against its own first run, the plain pipeline gives the same text for 28 of 30 questions, with 30 claims, all
found. The two pipelines differ a little more than one pipeline differs from itself (25 against 28 of 30),
in wording and one extra citation.

One more difference does not show here. LangChain's answer step sometimes cuts a quote with `...`
(*Denver linebacker Von Miller...recording five solo tackles*), which Lab 02's check accepts because both parts are in
the line, in order. On `gpt-4o-mini` it did so in 5 of our first 6 runs, and Lab 02's `answer()` in none of 12. The
messages are the same; the schema is not. LangChain writes it from the Pydantic class, with `$defs`, a title on every
field and another key order, and a schema is part of what the model reads: on that question, 11 of 20 calls with
LangChain's schema cut the quote, 0 of 20 with Lab 02's.

Last, the question Lab 02 used to test abstention, which the corpus cannot answer, three times in each
pipeline:

In [26]:
moon = "Who was the first person to walk on the Moon?"
moon_lines = evidence(store.similarity_search_with_score(moon, k=3))
plain_moon = [answering.answer(moon, moon_lines, settings)["answer"] for _ in range(3)]
lc_moon = [structured.invoke(messages_for(moon, moon_lines)).answer for _ in range(3)]
print(f"plain     abstained {plain_moon.count(INSUFFICIENT)} of 3 | {plain_moon[0]}")
print(f"LangChain abstained {lc_moon.count(INSUFFICIENT)} of 3 | {lc_moon[0]}")
print(f"\n{settings['CHAT_MODEL']} at {chat_url(settings)}")

plain     abstained 3 of 3 | Insufficient evidence.
LangChain abstained 3 of 3 | Insufficient evidence.

gpt-4o-mini at https://api.openai.com/v1


Both abstain in 3 of 3. The parity table for the whole pipeline, from this run:

| Measure | Plain Python (Labs 01 and 02) | LangChain |
|---|---|---|
| Known-source hit@3 on the 30 questions | 30 | 30 with `k=3` |
| Same chunks in the same order | (the reference) | 30 of 30 with `k=3`; 0 of 30 at the default `k=4` |
| Refused, of 30 answerable questions | 0 | 0 |
| Claims, and quotes found | 30, 30 | 31, 31 |
| The Moon question, abstained | 3 of 3 | 3 of 3 |

The framework changed the code (a `Document` for a dictionary, a store for the NumPy product, a Pydantic class for
the schema and `parse()`), not the retrieval, the refusals or the citations, once the one default that differed was
set. The schema it writes can still change the wording, such as a quote cut with `...`, which is why a parity check
counts every claim and repeats the run.

### Checkpoint F

| You should see | Our run |
|---|---|
| 30 questions | both refuse 0 of 30, every quote found (30 and 31 claims); same cited IDs 29 of 30, same text 25 of 30 |
| Plain, second run | 0 refused, 30 claims all found; same text as its first run 28 of 30 |
| The Moon question | *Insufficient evidence.* 3 of 3 (plain), 3 of 3 (LangChain) |

---

## Part G: Leave documents out with a metadata filter (self-study)

Part G uses the metadata kept in Part A: the store can leave documents out before it ranks.

A customer who uses Atlas asks *How quickly do reset links expire?* The question names no product. Search it and
answer it three times:

In [27]:
reset = "How quickly do reset links expire?"
reset_lines = evidence(store.similarity_search_with_score(reset, k=3))
print("evidence:", [line[1:line.index("]")] for line in reset_lines])
for _ in range(3):
    print(" ", structured.invoke(messages_for(reset, reset_lines)).answer)

evidence: ['H02', 'H04', 'H01']


  Atlas reset links expire after 30 minutes, while Boreal reset links expire after 10 minutes.


  Atlas reset links expire after 30 minutes, while Boreal reset links expire after 10 minutes.


  Atlas reset links expire after 30 minutes, while Boreal reset links expire after 10 minutes.


H02 comes first, then the Boreal policy H04, and all three answers give both products: *Atlas reset links expire after 30 minutes, while Boreal reset links expire after 10 minutes.* For an Atlas customer the Boreal policy is noise, and the question's words cannot keep it out: only the
customer's account says which product they use. A **metadata filter** can. In `InMemoryVectorStore` a filter is a
function that takes a `Document` and returns `True` to keep it, and the store applies it to every document before it
ranks. Each policy's title starts with its product's name, and no chunk title starts with *Atlas*:

In [28]:
def atlas_only(document):
    return document.metadata["title"].startswith("Atlas")

atlas_lines = evidence(store.similarity_search_with_score(reset, k=3, filter=atlas_only))
print("evidence:", [line[1:line.index("]")] for line in atlas_lines])
for _ in range(3):
    print(" ", structured.invoke(messages_for(reset, atlas_lines)).answer)

evidence: ['H02', 'H01', 'H05']


  Atlas reset links expire after 30 minutes.


  Atlas reset links expire after 30 minutes.


  Atlas reset links expire after 30 minutes.


H04 is gone and the Atlas billing policy H05 takes third place; all three answers give Atlas's 30 minutes only. The filter chose by who is asking, which a similarity score cannot. Lab 11 uses the same mechanism
for permissions, with the `access` field. A store that keeps its data in a database takes the filter in its own syntax,
often a dictionary, because the database applies it.

### Checkpoint G

| You should see | Our run |
|---|---|
| No filter | evidence H02, H04, H01; both products' times in 3 of 3 |
| `atlas_only` | evidence H02, H01, H05; *Atlas reset links expire after 30 minutes.* in 3 of 3 |

---

## Part H: Keep the pipeline in `shared/lc_pipeline.py` (self-study)

Part H moves the LangChain pipeline into one module, the baseline the next labs measure their changes against.

`shared/lc_pipeline.py` holds the steps of Parts A, C and E, with `k=3` set in the search and `JSON_MODE` read from
the settings. Its `load_corpus()` is Lab 04's `corpus()` from `shared/trace.py`, the same 277 records as Part A,
imported under the name the next labs use, so the corpus is defined in one place. Read it:

In [29]:
import shared.lc_pipeline

print(inspect.getsource(shared.lc_pipeline))

"""The pipeline of Labs 01 and 02, rebuilt with LangChain in Lab 05: the baseline for the labs after it.

load_corpus() is Lab 04's corpus(): the 272 chunks and the five policies. to_documents() turns its records into
LangChain Documents. build_store() embeds them into an InMemoryVectorStore. retrieve() searches it, with k=3 by
default (the store's own default is 4). evidence() writes the hits as "[ID] text" lines, and answer() asks for a
CitedAnswer with Lab 02's rules, so shared.answer.quote_found() checks its claims unchanged.
"""
from __future__ import annotations

from langchain_core.documents import Document
from langchain_core.vectorstores import InMemoryVectorStore
from pydantic import BaseModel

from shared.answer import RULES
from shared.settings import lc_chat_model, lc_embeddings
from shared.trace import corpus as load_corpus


class Claim(BaseModel):
    claim: str
    source_id: str
    quote: str


class CitedAnswer(BaseModel):
    answer: str
    claims: list[Claim]


d

Use it from the start, as a later lab will: corpus, documents and store, then search, evidence lines, answer and the
quote check for the Tesla and Moon questions.

In [30]:
from shared import lc_pipeline

pipeline_store = lc_pipeline.build_store(lc_pipeline.to_documents(lc_pipeline.load_corpus()), settings)
for q in (question, moon):
    q_lines = lc_pipeline.evidence(lc_pipeline.retrieve(pipeline_store, q))
    reply = lc_pipeline.answer(q, q_lines, settings)
    print(reply.answer, "| quotes found:", [answering.quote_found(claim.model_dump(), q_lines) for claim in reply.claims])

Nikola Tesla was born in the year 1856. | quotes found: [True]


Insufficient evidence. | quotes found: []


The module gives the same results: Tesla in 1856 with its quote found, and *Insufficient evidence.* for the Moon. From here on the labs call `load_corpus()`, `to_documents()`, `build_store()`, `retrieve()`, `evidence()` and
`answer()` by name, and keep Lab 02's `quote_found()` for the check.

### Checkpoint H

| You should see | Our run |
|---|---|
| Module results | *Nikola Tesla was born in the year 1856.*, quote found; *Insufficient evidence.* for the Moon |

---

## What you can do now

- Turn chunk records into `Document`s that keep their IDs and fields as metadata.
- Build LangChain's embeddings and chat model from `models.env` with `lc_embeddings()` and `lc_chat_model()`, and say
  why the embeddings send token IDs to OpenAI and text to other servers.
- Index documents in `InMemoryVectorStore` and search them with scores.
- Check a framework's results against your own question by question, and find a default in a function's signature.
- Get a cited answer as a Pydantic object with `with_structured_output`, and check its quotes with `quote_found()`.
- Leave documents out of a search with a metadata filter.

## Challenge

Use only what this lab showed. A worked solution is in `solved/` for the instructor. Task 1 is the challenge; task 2 is
optional. For each task, keep the printed results and your answer to **Explain**.

1. **Check a retriever against your own search.** Much LangChain code searches through a **retriever**, an object with
   `invoke()` and `batch()` that returns documents without scores. `store.as_retriever()` makes one from the store.
   Does it return what your `search()` returns?

   - Under **Task 1**, make a retriever with `store.as_retriever()`. The cell runs it on the 30 questions and counts
     the questions whose chunk IDs match `plain_ids` from Part D, in order.
   - If they do not all match, find out why from what the cell prints and the
     [`as_retriever` reference](https://reference.langchain.com/python/langchain-core/vectorstores/base/VectorStore),
     then fix the retriever and run the cell again.

   **Check:** the count before and after your fix. Our worked solution got 0 of 30, then 30 of 30.

   **Explain:** which check in this lab would have missed the difference, and which one caught it?

2. **Optional: write a filter for another customer.** Complete `boreal_only` under **Task 2** for a customer who uses
   Boreal. The cell asks Part G's question and one the Boreal policy does not answer, *How are Boreal invoices sent?*

   **Check:** for each question, the evidence IDs and the answer. Our worked solution got `['H04']` and *Boreal reset links expire after 10 minutes.* for Part G's question, and `['H04']` and *Insufficient evidence.* for the invoices question.

   **Explain:** how many documents did the store return for each question, and why? What would the model have had to
   go on without the abstention line?

**Task 1.** Your retriever, then the count:

In [31]:
retriever = None    # TODO: a retriever from store.as_retriever()

if retriever is None:
    print("⚠️ Make a retriever, then run the cell again.")
else:
    retriever_ids = [[doc.id for doc in docs] for docs in retriever.batch([q["question"] for q in questions])]
    print(f"same chunks in the same order: {sum(a == b for a, b in zip(plain_ids, retriever_ids))} of {len(questions)}")
    print("chunks per question:", len(retriever_ids[0]), "| search_kwargs:", retriever.search_kwargs)

⚠️ Make a retriever, then run the cell again.


**Task 2 (optional).** Complete `boreal_only`, then run the cell:

In [32]:
def boreal_only(document):
    # TODO: return True for documents about Boreal (see atlas_only in Part G)
    return None

if boreal_only(documents[-1]) is None:
    print("⚠️ Complete boreal_only, then run the cell again.")
else:
    for q in (reset, "How are Boreal invoices sent?"):
        q_lines = evidence(store.similarity_search_with_score(q, k=3, filter=boreal_only))
        print([line[1:line.index("]")] for line in q_lines], "|", structured.invoke(messages_for(q, q_lines)).answer)

⚠️ Complete boreal_only, then run the cell again.


## Troubleshooting

| Symptom | Likely cause | Fix |
|---|---|---|
| `ModuleNotFoundError: No module named 'langchain_community'` | code from an older tutorial; these labs install only LangChain's core packages | Import the store from `langchain_core.vectorstores` and the models from `langchain_openai`, as this lab does |
| `OpenAIError: Missing credentials. Please pass an api_key ...` | no key in the environment when the model or embeddings object was built | Add the key as Lab 00's README shows for your route, run Config, then build the object again |
| `OpenAIInvalidRequestError: Error code: 400 ... 'temperature' does not support 0.0 with this model.` | `ChatOpenAI(temperature=0)` built by hand for a reasoning model such as `gpt-6-luna` | Build models with `lc_chat_model()`, and set `REASONING_EFFORT=none` for `gpt-6-luna` |
| `ValueError: Unsupported function` followed by a schema | Lab 02's `SCHEMA` dictionary passed to `with_structured_output`, which needs a name for a dictionary schema | Pass the `CitedAnswer` class, or `{"name": "cited_answer", "strict": True, "schema": SCHEMA}` |
| `KeyError: 'access'` during a filtered search | the filter reads a field only some documents have (only the policies have `access`) | Read it with `document.metadata.get("access")` |

## Resources

- [LangChain: build a semantic search engine](https://docs.langchain.com/oss/python/langchain/knowledge-base): documents,
  embeddings, vector stores and retrievers, the steps of Parts A to D
- [LangChain: models, structured output](https://docs.langchain.com/oss/python/langchain/models#structured-output):
  `with_structured_output` with a Pydantic class, its methods, and `include_raw=True` for the token counts
- [LangChain reference: `InMemoryVectorStore`](https://reference.langchain.com/python/langchain-core/vectorstores/in_memory/InMemoryVectorStore):
  every search method, its defaults and the filter argument
- [LangChain reference: `OpenAIEmbeddings`](https://reference.langchain.com/python/langchain-openai/embeddings/base/OpenAIEmbeddings):
  `check_embedding_ctx_length` and the settings for other OpenAI-compatible servers
- [Pydantic: models](https://docs.pydantic.dev/latest/concepts/models/): fields, validation and `model_dump()`

Next: [Lab 06](../06_Evaluation_Set/README.md), *How to build an evaluation set for RAG*. Back to [the lab list](../README.md).